In [1]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from typing import Dict, List, Optional
import pickle
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

In [2]:
class BusinessReviewEmbedderFromCSV:
    """
    Creates weighted business embeddings from review CSV using star ratings.
    """
    
    def __init__(self, model_name: str = 'all-MiniLM-L6-v2'):
        """
        Args:
            model_name: SentenceTransformer model to use
        """
        print(f"Loading model: {model_name}")
        self.model = SentenceTransformer(model_name)
        self.embedding_dim = self.model.get_sentence_embedding_dimension()
        print(f"Embedding dimension: {self.embedding_dim}")
        
    def load_data(self, csv_path: str) -> pd.DataFrame:
        """Load and inspect the review data"""
        df = pd.read_csv(csv_path)
        print(f"\nLoaded {len(df):,} reviews")
        print(f"Unique businesses: {df['business_id'].nunique():,}")
        print(f"Star distribution:\n{df['stars'].value_counts().sort_index()}")
        
        # Check for missing values
        missing = df[['business_id', 'stars', 'text']].isnull().sum()
        if missing.any():
            print(f"\n⚠️ Missing values detected:\n{missing}")
            df = df.dropna(subset=['business_id', 'stars', 'text'])
            print(f"Cleaned to {len(df):,} reviews")
            
        return df
    
    def _star_weight_function(self, 
                             stars: float, 
                             method: str = 'extremity',
                             useful_col: Optional[pd.Series] = None) -> float:
        """
        Convert star rating to importance weight.
        
        Args:
            stars: Rating from 1-5
            method: Weighting strategy
                - 'linear': w = stars/5
                - 'extremity': Weight by distance from neutral (3 stars)
                - 'sigmoid': Sigmoid centered at 3 stars
                - 'combined': Combine stars + useful count
            useful_col: Optional 'useful' column values for combined weighting
        """
        if method == 'linear':
            return stars / 5.0
        
        elif method == 'extremity':
            # Emphasize extreme ratings (1-star and 5-star matter more)
            return 0.5 + 0.5 * (abs(stars - 3) / 2)  # Range [0.5, 1.0]
        
        elif method == 'sigmoid':
            # Non-linear emphasis on extremes
            x = stars - 3  # Center around neutral
            return 1 / (1 + np.exp(-x))  # Range [~0.12, ~0.88]
        
        elif method == 'combined' and useful_col is not None:
            # Combine star extremity with usefulness score
            star_weight = 0.5 + 0.5 * (abs(stars - 3) / 2)
            # Log-scale useful to avoid dominance by viral reviews
            useful_weight = np.log1p(useful_col) / np.log1p(useful_col.max())
            return 0.7 * star_weight + 0.3 * useful_weight
        
        else:
            return 1.0  # Equal weight fallback
    
    def create_business_embeddings(self, 
                                   df: pd.DataFrame,
                                   weight_method: str = 'extremity',
                                   batch_size: int = 32,
                                   use_useful: bool = False) -> Dict[str, Dict]:
        """
        Generate weighted average embeddings per business.
        
        Returns:
            Dictionary with business_id as key and dict with:
                - 'embedding': Normalized weighted average vector
                - 'review_count': Number of reviews
                - 'avg_stars': Average star rating
                - 'weighted_avg_stars': Weight-accounted average stars
        """
        business_groups = df.groupby('business_id')
        business_embeddings = {}
        
        print(f"\nProcessing {business_groups.ngroups} businesses...")
        print(f"Weighting method: {weight_method}")
        
        for business_id, group in tqdm(business_groups, desc="Encoding businesses"):
            reviews = group['text'].tolist()
            stars = group['stars'].values
            
            if len(reviews) == 0:
                continue
            
            # Generate embeddings for all reviews of this business
            review_embeddings = self.model.encode(
                reviews, 
                batch_size=batch_size,
                show_progress_bar=False,
                convert_to_numpy=True
            )
            
            # Calculate weights
            if weight_method == 'combined' and use_useful and 'useful' in group.columns:
                weights = np.array([
                    self._star_weight_function(s, method=weight_method, useful_col=u)
                    for s, u in zip(stars, group['useful'].values)
                ])
            else:
                weights = np.array([
                    self._star_weight_function(s, method=weight_method)
                    for s in stars
                ])
            
            # Weighted average
            if weights.sum() > 0:
                weighted_avg = np.average(review_embeddings, axis=0, weights=weights)
            else:
                weighted_avg = np.mean(review_embeddings, axis=0)
            
            # L2 normalization for cosine similarity
            weighted_avg = weighted_avg / np.linalg.norm(weighted_avg)
            
            # Also compute unweighted average for comparison
            unweighted_avg = np.mean(review_embeddings, axis=0)
            unweighted_avg = unweighted_avg / np.linalg.norm(unweighted_avg)
            
            business_embeddings[business_id] = {
                'embedding': weighted_avg,
                'unweighted_embedding': unweighted_avg,
                'review_count': len(reviews),
                'avg_stars': stars.mean(),
                'weighted_avg_stars': np.average(stars, weights=weights) if weights.sum() > 0 else stars.mean(),
                'total_useful': group['useful'].sum() if 'useful' in group.columns else 0
            }
        
        print(f"\n✓ Generated embeddings for {len(business_embeddings)} businesses")
        return business_embeddings
    
    def save_embeddings(self, embeddings: Dict, output_path: str):
        """Save embeddings to pickle file"""
        with open(output_path, 'wb') as f:
            pickle.dump(embeddings, f)
        print(f"✓ Saved embeddings to {output_path}")
    
    def load_embeddings(self, input_path: str) -> Dict:
        """Load saved embeddings"""
        with open(input_path, 'rb') as f:
            return pickle.load(f)
    
    def analyze_results(self, embeddings: Dict):
        """Print statistics about the generated embeddings"""
        print("\n" + "="*50)
        print("EMBEDDING STATISTICS")
        print("="*50)
        
        review_counts = [e['review_count'] for e in embeddings.values()]
        avg_stars = [e['avg_stars'] for e in embeddings.values()]
        
        print(f"Total businesses: {len(embeddings):,}")
        print(f"Avg reviews per business: {np.mean(review_counts):.1f}")
        print(f"Median reviews per business: {np.median(review_counts):.0f}")
        print(f"Min/Max reviews: {min(review_counts)}/{max(review_counts)}")
        print(f"\nAvg star rating across businesses: {np.mean(avg_stars):.2f}")
        
        # Find businesses with highest/lowest star variance
        # (would need original df for this)
    
    def find_similar_businesses(self, 
                                embeddings: Dict, 
                                business_id: str, 
                                top_k: int = 5) -> List[tuple]:
        """Find most similar businesses using cosine similarity"""
        if business_id not in embeddings:
            return []
        
        query_emb = embeddings[business_id]['embedding']
        similarities = []
        
        for bid, data in embeddings.items():
            if bid != business_id:
                sim = np.dot(query_emb, data['embedding'])
                similarities.append((bid, sim, data['avg_stars']))
        
        similarities.sort(key=lambda x: x[1], reverse=True)
        return similarities[:top_k]


In [ ]:


# Initialize embedder
embedder = BusinessReviewEmbedderFromCSV(model_name='all-MiniLM-L6-v2')

# Load your data
df = embedder.load_data('data_review_useful.csv')


Loading model: all-MiniLM-L6-v2


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

In [ ]:

# Create embeddings with different weighting strategies
strategies = {
    'extremity': 'Weight by distance from neutral (3 stars)',
    'linear': 'Linear weight based on stars',
    'combined': 'Stars + useful count'
}

for method, description in strategies.items():
    print(f"\n{'='*60}")
    print(f"METHOD: {method}")
    print(f"Description: {description}")
    print('='*60)
    
    # Generate embeddings
    embeddings = embedder.create_business_embeddings(
        df, 
        weight_method=method,
        use_useful=(method == 'combined')
    )
    
    # Save
    embedder.save_embeddings(embeddings, f'business_embeddings_{method}.pkl')
    
    # Quick analysis
    embedder.analyze_results(embeddings)
    
    # Example: Find similar businesses for a sample business
    sample_business = list(embeddings.keys())[0]
    print(f"\nSample similar businesses for {sample_business}:")
    similar = embedder.find_similar_businesses(embeddings, sample_business, top_k=3)
    for bid, sim, stars in similar:
        print(f"  - {bid}: similarity={sim:.3f}, avg_stars={stars:.1f}")
